In [0]:
%pip install "numpy<2.0" --quiet
%pip install faiss-cpu==1.8.0 --quiet
%pip install langgraph==0.2.27 langchain==0.3.7 --quiet
dbutils.library.restartPython()

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 06 — LangGraph Multi-Agent Orchestrator (v4 — All Bugs Fixed)
# MAGIC
# MAGIC ## Bug Fixes Applied in v4
# MAGIC
# MAGIC | ID | Bug | Fix |
# MAGIC |----|-----|-----|
# MAGIC | B1 | `_clean_sql()` never returns SELECT match — always returns raw string | Return `match.group(1)` when match found |
# MAGIC | B2 | SQL schema lists wrong column names (operatortypeid vs operatorTypeId, enhanced_* in IDP, total_anomaly_flags in IDP) | Corrected all column names against real CSVs |
# MAGIC | B3 | LLM generates `infectiousDisease` (missing 's'), `gynecologyObstetrics` (missing And) | Added `_normalize_sql_column_names()` post-processing |
# MAGIC | B4 | OR without parentheses causes wrong SQL logic | SQL prompt now enforces parentheses rule |
# MAGIC | B5 | `GROUP BY procedure_enriched` (JSON string) fails for Q7.5 | SQL prompt blocks GROUP BY on JSON cols; rewritten query approach |
# MAGIC | B6 | Multi-agent chaining: `_wrap_node` defined but never applied; sub_agents never consumed | Each node now pops its own key from sub_agents in return dict |
# MAGIC | B7 | Q7.6 "oversupply" routes to sql instead of medical/desert | Added `oversupply` and `scarcity.*complex` to medical heuristics |
# MAGIC | B8 | Hardcoded Databricks token | Use dbutils.secrets with fallback chain |
# MAGIC | B9 | `gold_anomaly_report` may not exist — no safe fallback | Try/except with None result, skip in synthesiser |
# MAGIC | B10 | Geo node city extraction bug — coords not used after found | Explicit coord assignment after center_name lookup |
# MAGIC | B11 | SQL preamble "python\\n" not stripped | Regex now extracts first SELECT..END block |
# MAGIC | B12 | Synthesiser references planning_data["plan"] but key may be missing | Safe `.get()` access |

# COMMAND ----------
# MAGIC %pip install "numpy<2.0" faiss-cpu==1.8.0 langgraph==0.2.27 langchain==0.3.7 --quiet

# COMMAND ----------

# COMMAND ----------
# MAGIC %md ## 0. Imports & Configuration

# COMMAND ----------

import json, os, re, time, math, pickle
import operator
from typing import TypedDict, Annotated, List, Optional, Dict, Any, Tuple
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests
import mlflow
import faiss

from langgraph.graph import StateGraph, END
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.getOrCreate()
print(f"Run: {datetime.now(timezone.utc).isoformat()}")

# COMMAND ----------

class AgentConfig:
    DATABRICKS_HOST = spark.conf.get("spark.databricks.workspaceUrl", "").rstrip("/")

    # B8 FIX: Never hardcode token — use secrets with fallback chain
    @staticmethod
    def _get_token() -> str:
        # 1. Databricks Secrets (preferred in production)
        try:
            return dbutils.secrets.get(scope="virtue-foundation", key="databricks-token")
        except Exception:
            pass
        # 2. Context token (works in any Databricks notebook)
        try:
            return dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
        except Exception:
            pass
        # 3. Environment variable (for local testing)
        return os.getenv("DATABRICKS_TOKEN", "")

    DATABRICKS_TOKEN: str = ""   # Set in __post_init__ equivalent below

    LLM_MODEL    = "databricks-meta-llama-3-3-70b-instruct"
    LLM_ENDPOINT: str = ""
    EMBED_ENDPOINT: str = ""
    EMBED_DIM    = 1024

    IDP_TABLE      = "virtue_foundation.ghana.gold_idp_enriched"
    REGIONAL_TABLE = "virtue_foundation.ghana.gold_regional_summary"
    GOLD_FALLBACK  = "virtue_foundation.ghana.gold_facilities_enriched"
    ANOMALY_TABLE  = "virtue_foundation.ghana.gold_anomaly_flags"
    DESERT_TABLE   = "virtue_foundation.ghana.gold_medical_desert_scores"
    ANOMALY_REPORT = "virtue_foundation.ghana.gold_anomaly_report"

    RAG_DIR        = "/Workspace/Users/dasdeepayan08@gmail.com/databricks_accenture_hackathon_virtue_foundationtrack/databricks/rag"
    RAG_INDEX_PATH: str = ""
    RAG_META_PATH: str  = ""
    VOLUME_INDEX   = "/Volumes/virtue_foundation/ghana/rag/faiss_index.bin"
    VOLUME_META    = "/Volumes/virtue_foundation/ghana/rag/faiss_metadata.json"

    MLFLOW_EXP = "/Users/dasdeepayan08@gmail.com/virtue-foundation-idp-hackathon"

    GHANA_CITY_COORDS: Dict[str, Tuple[float, float]] = {
        "accra":        (5.5600, -0.1969), "tema":          (5.6698, -0.0166),
        "kumasi":       (6.6884, -1.6244), "takoradi":      (4.8978, -1.7561),
        "tamale":       (9.4035, -0.8393), "cape coast":    (5.1053, -1.2466),
        "ho":           (6.6008,  0.4703), "koforidua":     (6.0844, -0.2554),
        "sunyani":      (7.3349, -2.3272), "wa":            (10.0601, -2.5099),
        "bolgatanga":   (10.7856, -0.8514), "techiman":     (7.5811, -1.9356),
        "berekum":      (7.4544, -2.5857), "nkawkaw":       (6.5383, -0.7673),
        "obuasi":       (6.2000, -1.6667), "tarkwa":        (5.3000, -1.9833),
        "sekondi":      (4.9374, -1.7073), "winneba":       (5.3525, -0.6267),
        "navrongo":     (10.8942, -1.0940), "bawku":        (11.0594, -0.2428),
        "lawra":        (10.6547, -2.9064), "damongo":      (9.0802, -1.8215),
        "salaga":       (8.5575, -0.5154), "nkoranza":      (7.5667, -1.7167),
        "goaso":        (6.7971, -2.5191), "savelugu":      (9.6246, -0.8328),
        "yendi":        (9.4427, -0.0081), "wa":            (10.0601, -2.5099),
    }


cfg = AgentConfig()
cfg.DATABRICKS_TOKEN = AgentConfig._get_token()
cfg.LLM_ENDPOINT  = f"https://{cfg.DATABRICKS_HOST}/serving-endpoints/{cfg.LLM_MODEL}/invocations"
cfg.EMBED_ENDPOINT = f"https://{cfg.DATABRICKS_HOST}/serving-endpoints/databricks-bge-large-en/invocations"
cfg.RAG_INDEX_PATH = f"{cfg.RAG_DIR}/faiss_index.bin"
cfg.RAG_META_PATH  = f"{cfg.RAG_DIR}/faiss_metadata.json"

print(f"LLM  : {cfg.LLM_ENDPOINT}")
print(f"Token: {'✅ set' if cfg.DATABRICKS_TOKEN else '❌ MISSING'}")

# COMMAND ----------
# MAGIC %md ## 1. Shared Utilities

# COMMAND ----------

def call_llama(messages: List[Dict], system_prompt: Optional[str] = None,
               max_tokens: int = 1024, temperature: float = 0.1,
               retries: int = 3) -> str:
    full: List[Dict] = []
    if system_prompt:
        full.append({"role": "system", "content": system_prompt})
    full.extend(messages)
    headers = {"Authorization": f"Bearer {cfg.DATABRICKS_TOKEN}", "Content-Type": "application/json"}
    payload = {"messages": full, "max_tokens": max_tokens, "temperature": temperature,
               "top_p": 0.95, "stream": False}
    for attempt in range(retries):
        try:
            r = requests.post(cfg.LLM_ENDPOINT, headers=headers, json=payload, timeout=120)
            r.raise_for_status()
            return r.json()["choices"][0]["message"]["content"]
        except requests.HTTPError as e:
            code = getattr(e.response, "status_code", None)
            if code == 429:
                wait = 2 ** attempt * 5
                print(f"  Rate-limit. Waiting {wait}s"); time.sleep(wait)
            elif attempt == retries - 1:
                return f"[LLM error {code}: {str(e)[:150]}]"
            else:
                time.sleep(3 * (attempt + 1))
        except Exception as e:
            if attempt == retries - 1:
                return f"[LLM error: {str(e)[:150]}]"
            time.sleep(3 * (attempt + 1))
    return ""


def parse_json_llm(raw: str) -> Dict:
    if not raw:
        return {}
    clean = re.sub(r"^```(?:json)?\s*", "", raw.strip(), flags=re.IGNORECASE)
    clean = re.sub(r"\s*```\s*$", "", clean).strip()
    s, e  = clean.find("{"), clean.rfind("}")
    if s != -1 and e != -1:
        clean = clean[s:e + 1]
    try:
        return json.loads(clean)
    except Exception:
        clean = re.sub(r",\s*}", "}", clean)
        clean = re.sub(r",\s*]", "]", clean)
        try:
            return json.loads(clean)
        except Exception:
            return {}


def ensure_list(x) -> List[str]:
    if x is None:
        return []
    if isinstance(x, float) and (x != x):
        return []
    if isinstance(x, (list, tuple)):
        return [str(v) for v in x if v is not None and str(v).strip() not in ("", "null", "nan")]
    if isinstance(x, str):
        s = x.strip()
        if s in ("", "null", "[]", "nan", "None"):
            return []
        try:
            p = json.loads(s)
            if isinstance(p, list):
                return [str(v) for v in p if v]
            return [str(p)]
        except Exception:
            return [s]
    return [str(x)] if x else []


def safe_str(v, d: str = "") -> str:
    if v is None: return d
    s = str(v).strip()
    return d if s in ("None", "nan", "null", "") else s


def safe_float(v, d: float = 0.0) -> float:
    try:
        return float(v) if v is not None and str(v) not in ("nan", "None", "null", "") else d
    except Exception:
        return d


def safe_int(v, d: int = 0) -> int:
    try:
        return int(float(v)) if v is not None and str(v) not in ("nan", "None", "null", "") else d
    except Exception:
        return d


def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return R * 2 * math.asin(math.sqrt(a))

# COMMAND ----------
# MAGIC %md ## 2. Agent State

# COMMAND ----------

class StepCitation(TypedDict):
    step_name    : str
    input_data   : str
    output_data  : str
    data_sources : List[str]
    confidence   : float


class AgentState(TypedDict):
    query            : str
    session_id       : str
    query_type       : str
    sub_agents       : List[str]   # remaining sub-agents to execute
    sql_results      : Optional[str]
    rag_results      : List[Dict]
    geo_results      : Optional[Dict]
    map_data         : Optional[Dict]
    anomalies        : List[Dict]
    desert_data      : Optional[Dict]
    medical_analysis : Optional[Dict]
    planning_data    : Optional[Dict]
    ngo_data         : Optional[Dict]
    answer           : str
    citations        : List[Dict]
    step_citations   : List[StepCitation]
    steps            : Annotated[List[str], operator.add]
    run_id           : str
    error            : Optional[str]

# COMMAND ----------
# MAGIC %md ## 3. SQL Schema (B2 Fixed: Verified against all real CSVs)

# COMMAND ----------

# B2 FIX: All column names verified against real CSVs
SQL_SCHEMA = """
=== virtue_foundation.ghana — VERIFIED SCHEMA (v4) ===

TABLE: virtue_foundation.ghana.gold_idp_enriched (~912 rows)
  -- Identity --
  unique_id STRING, name STRING, source_url STRING
  -- Facility type (use THESE exact column names) --
  facilityTypeId STRING               -- original: 'hospital','clinic',etc  (camelCase)
  facility_type_clean STRING          -- cleaned: 'hospital','clinic','pharmacy','dentist','doctor'
  operatorTypeId STRING               -- 'public','private' (camelCase — NOT operatortypeid)
  organization_type_clean STRING      -- 'facility','ngo'
  -- Location --
  city_clean STRING, region_normalised STRING
  address_line1 STRING
  latitude DOUBLE, longitude DOUBLE
  -- Numerics (NULL for most rows — always COALESCE) --
  number_doctors_int DOUBLE           -- COALESCE(number_doctors_int, 0)
  capacity_int DOUBLE                 -- COALESCE(capacity_int, 0)
  -- Clinical boolean flags --
  has_emergency_medicine BOOLEAN, has_obstetrics BOOLEAN, has_surgery BOOLEAN
  has_pediatrics BOOLEAN, has_icu BOOLEAN, has_radiology BOOLEAN
  has_infectious_disease BOOLEAN, has_mental_health BOOLEAN
  -- Facility type boolean flags --
  is_hospital BOOLEAN, is_clinic BOOLEAN, is_ngo BOOLEAN
  is_public BOOLEAN, is_private BOOLEAN
  -- Counts --
  procedure_count BIGINT, equipment_count BIGINT, capability_count BIGINT
  specialty_count BIGINT
  -- IDP arrays (JSON STRING — use LIKE for substring search; CANNOT use GROUP BY on these) --
  specialties_enriched STRING, procedure_enriched STRING
  equipment_enriched STRING, capability_enriched STRING
  -- Native arrays --
  specialties_parsed ARRAY<STRING>
  capability_parsed ARRAY<STRING>, procedure_parsed ARRAY<STRING>, equipment_parsed ARRAY<STRING>
  -- Anomaly flags (stat_ = statistical; NOTE: no enhanced_* columns here) --
  stat_anomaly_capability_inflation BOOLEAN
  stat_anomaly_hospital_no_doctors BOOLEAN
  stat_anomaly_clinic_claims_icu BOOLEAN
  stat_anomaly_ghost_facility BOOLEAN
  stat_anomaly_specialty_mismatch BOOLEAN
  stat_anomaly_procedure_breadth BOOLEAN
  total_stat_anomalies BIGINT         -- use this (NOT total_anomaly_flags) for IDP table
  -- Quality --
  capability_is_valid BOOLEAN, capability_confidence DOUBLE
  data_completeness_score DOUBLE, is_rag_ready BOOLEAN
  -- Desert --
  medical_desert_score DOUBLE, desert_label STRING
  -- IDP --
  idp_citations ARRAY<STRING>, idp_run_id STRING
  ngo_serves_ghana BOOLEAN
  accepts_volunteers_bool BOOLEAN

TABLE: virtue_foundation.ghana.gold_regional_summary (17 rows)
  region_normalised STRING
  total_facilities BIGINT, clinical_facility_count BIGINT
  hospital_count BIGINT, clinical_hospital_count BIGINT, clinic_count BIGINT
  public_facilities BIGINT, private_facilities BIGINT, ngo_count BIGINT
  avg_doctors DOUBLE, total_doctors BIGINT
  avg_bed_capacity DOUBLE, total_beds BIGINT
  avg_completeness DOUBLE
  emergency_medicine_facilities BIGINT, obstetrics_facilities BIGINT
  surgery_facilities BIGINT, pediatrics_facilities BIGINT
  icu_facilities BIGINT, infectious_disease_facilities BIGINT
  radiology_facilities BIGINT, mental_health_facilities BIGINT
  facilities_with_procedures BIGINT, facilities_with_equipment BIGINT
  facilities_with_capabilities BIGINT, volunteer_facilities BIGINT
  procedure_breadth_anomalies BIGINT, rag_ready_count BIGINT
  region_centroid_lat DOUBLE, region_centroid_lon DOUBLE
  total_region_anomalies BIGINT
  all_specialties ARRAY<STRING>               -- native array, use array_contains()
  missing_critical_specialties ARRAY<STRING>  -- native array
  critical_specialty_gap_count INT
  recommended_actions ARRAY<STRING>
  medical_desert_score FLOAT, desert_label STRING

TABLE: virtue_foundation.ghana.gold_medical_desert_scores (17 rows)
  region STRING                               -- use region (NOT region_normalised)
  total_facilities INT, hospital_count INT, ngo_count INT
  total_beds INT, total_doctors INT
  population_estimate INT
  facilities_per_100k FLOAT, beds_per_10k FLOAT, doctors_per_10k FLOAT
  hospitals_per_100k FLOAT
  critical_specialties_covered INT
  critical_specialties_missing STRING         -- JSON string list
  covered_specialty_names STRING              -- JSON string list
  density_component FLOAT, specialist_component FLOAT
  infrastructure_component FLOAT, completeness_component FLOAT
  anomaly_penalty FLOAT
  medical_desert_score FLOAT
  mds_label STRING     -- CORRECT label col: 'Critical Desert','Severe Desert','Moderate Desert','At Risk','Adequately Served'
  centroid_lat FLOAT, centroid_lon FLOAT
  recommended_actions STRING                  -- JSON string
  avg_data_completeness FLOAT

TABLE: virtue_foundation.ghana.gold_anomaly_flags (909 rows)
  unique_id STRING, name STRING, city_clean STRING
  region_normalised STRING, facility_type_clean STRING, organization_type_clean STRING
  latitude DOUBLE, longitude DOUBLE
  number_doctors_int DOUBLE, capacity_int DOUBLE
  data_completeness_score DOUBLE, capability_confidence DOUBLE
  capability_is_valid BOOLEAN
  has_emergency_medicine BOOLEAN, has_surgery BOOLEAN, has_icu BOOLEAN
  has_obstetrics BOOLEAN, has_radiology BOOLEAN, has_infectious_disease BOOLEAN, has_mental_health BOOLEAN
  procedure_count BIGINT, equipment_count BIGINT, capability_count BIGINT
  stat_anomaly_capability_inflation BOOLEAN, stat_anomaly_hospital_no_doctors BOOLEAN
  stat_anomaly_clinic_claims_icu BOOLEAN, stat_anomaly_ghost_facility BOOLEAN
  stat_anomaly_specialty_mismatch BOOLEAN, stat_anomaly_procedure_breadth BOOLEAN
  -- Enhanced flags (ONLY in gold_anomaly_flags, NOT in gold_idp_enriched) --
  enhanced_type_capability_mismatch BOOLEAN, enhanced_ghost_hospital BOOLEAN
  enhanced_procedures_no_equipment BOOLEAN, enhanced_low_idp_confidence BOOLEAN
  enhanced_suspicious_completeness BOOLEAN, enhanced_icu_no_infrastructure BOOLEAN
  total_anomaly_flags INT          -- use this in gold_anomaly_flags (NOT total_stat_anomalies)
  anomaly_risk_level STRING        -- 'CRITICAL','HIGH','MEDIUM','LOW','CLEAN'
  llm_priority_action STRING
  llm_data_quality_score FLOAT     -- 0.0–10.0
  llm_confirmed_anomaly_count INT
  llm_anomaly_severity STRING      -- 'CRITICAL','HIGH','MEDIUM','LOW','FALSE_POSITIVE'
  llm_clinical_assessment STRING
  llm_false_positive_reason STRING
  specialties_enriched STRING, procedure_enriched STRING
  equipment_enriched STRING, capability_enriched STRING
  capability_anomalies STRING
  medical_desert_score FLOAT, desert_label STRING
"""

# COMMAND ----------
# MAGIC %md ## 4. SQL Utilities (B1 Fix: _clean_sql now properly extracts SELECT)

# COMMAND ----------

# B1 FIX: The old version computed match but ALWAYS returned cleaned.strip()
# Now we properly return match.group(1) when SELECT is found
def _clean_sql(raw: str) -> str:
    """
    Extract valid SQL from LLM output.
    Handles: markdown fences, 'python\\n' prefix, preamble text, trailing explanation.
    B1 FIX: now actually returns the matched SELECT block.
    """
    if not raw:
        return ""
    # Strip markdown code fences
    clean = re.sub(r"^```(?:sql|python)?\s*", "", raw.strip(), flags=re.IGNORECASE)
    clean = re.sub(r"\s*```\s*$", "", clean).strip()
    # Strip inline SQL comments
    clean = re.sub(r"--[^\n]*", "", clean)
    # Strip leading language markers like "python", "sql", "Here is the SQL:"
    clean = re.sub(r"^(?:python|sql|Here\s+is.*?:|Note:)[^\n]*\n", "", clean,
                   flags=re.IGNORECASE | re.MULTILINE)
    # B1 FIX: Extract the first SELECT...LIMIT block
    match = re.search(r"(SELECT\b.+?)(?=\n\n|\Z)", clean, re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(1).strip()
    # Fallback: find anything starting with SELECT
    match2 = re.search(r"(SELECT\b.*)", clean, re.IGNORECASE | re.DOTALL)
    if match2:
        return match2.group(1).strip()
    return clean.strip()


# B3 FIX: Normalize specialty names that LLM commonly gets wrong
_SPECIALTY_NORM: Dict[str, str] = {
    # Missing 's'
    "'infectiousDisease'":              "'infectiousDiseases'",
    '"infectiousDisease"':              '"infectiousDiseases"',
    # Missing 'And'
    "'gynecologyObstetrics'":           "'gynecologyAndObstetrics'",
    "'gynecology_obstetrics'":          "'gynecologyAndObstetrics'",
    # Lowercase
    "'internalmedicine'":               "'internalMedicine'",
    "'generalsurgery'":                 "'generalSurgery'",
    "'emergencymedicine'":              "'emergencyMedicine'",
    # Alternative forms
    "'ObGyn'":                          "'gynecologyAndObstetrics'",
    "'ob-gyn'":                         "'gynecologyAndObstetrics'",
    "'critical care'":                  "'criticalCareMedicine'",
    "'criticalcare'":                   "'criticalCareMedicine'",
    # Specific IDP output normalization
    "'infectious_disease'":             "'infectiousDiseases'",
    "has_infectious_disease = true":    "has_infectious_disease = true",  # no change — column OK
}

def _normalize_sql_column_names(sql: str) -> str:
    """
    B3 FIX: Post-process LLM SQL to fix common specialty name mistakes.
    Also fix OR without parentheses.
    """
    # Fix specialty name typos
    for wrong, right in _SPECIALTY_NORM.items():
        sql = sql.replace(wrong, right)

    # B4 FIX: Wrap loose OR conditions in parentheses when inside WHERE
    # Pattern: "AND array_contains(...) OR col LIKE" → "AND (array_contains(...) OR col LIKE)"
    sql = re.sub(
        r"\bAND\s+(array_contains\([^)]+\))\s+OR\s+([\w_]+\s+LIKE\s+'[^']+')",
        r"AND (\1 OR \2)",
        sql, flags=re.IGNORECASE
    )
    # Fix operatortypeid → operatorTypeId (case correction)
    sql = re.sub(r'\boperatortypeid\b', 'operatorTypeId', sql, flags=re.IGNORECASE)

    return sql


# B5 FIX: Intercept GROUP BY on JSON string columns and rewrite query
def _guard_json_group_by(sql: str) -> Optional[str]:
    """
    B5 FIX: Detect GROUP BY on JSON-string columns (procedure_enriched, etc.)
    and substitute a safe alternative query using procedure_count.
    Returns corrected SQL or None if no fix needed.
    """
    JSON_COLS = ['procedure_enriched', 'equipment_enriched', 'capability_enriched',
                 'specialties_enriched', 'capability_parsed', 'specialties_parsed']
    for col in JSON_COLS:
        if re.search(rf'\bGROUP\s+BY\s+{col}\b', sql, re.IGNORECASE):
            # Replace with a safe count-based approach
            return f"""
SELECT
    region_normalised,
    SUM(CASE WHEN procedure_count <= 2 THEN 1 ELSE 0 END) AS facilities_few_procedures,
    SUM(CASE WHEN procedure_count >= 8 THEN 1 ELSE 0 END) AS facilities_many_procedures,
    AVG(procedure_count) AS avg_procedure_count,
    MAX(procedure_count) AS max_procedure_count,
    COUNT(*) AS total_facilities
FROM virtue_foundation.ghana.gold_idp_enriched
WHERE is_hospital = true OR is_clinic = true
GROUP BY region_normalised
ORDER BY facilities_few_procedures DESC
LIMIT 50
""".strip()
    return None

# COMMAND ----------
# MAGIC %md ## 5. SQL Generation System Prompt (B4 Fixed: parentheses enforcement)

# COMMAND ----------

SQL_GEN_SYSTEM_PROMPT = f"""You are a Spark SQL expert for Ghana healthcare intelligence.
Generate ONLY valid PySpark SQL. No markdown, no explanation, no code blocks.
Start your response directly with SELECT.

{SQL_SCHEMA}

CRITICAL SQL RULES — MUST FOLLOW EVERY TIME:
1. Boolean columns: WHERE is_hospital = true   (NOT 'true', NOT = 1)
2. Native ARRAY columns: array_contains(specialties_parsed, 'emergencyMedicine')
3. JSON STRING columns (procedure_enriched, capability_enriched, specialties_enriched):
   Use LIKE for search: WHERE specialties_enriched LIKE '%cardiology%'
   NEVER use GROUP BY on JSON string columns — they are not scalars
4. ALWAYS use COALESCE for nullable numerics: COALESCE(number_doctors_int, 0)
5. OR MUST have parentheses: WHERE is_hospital = true AND (array_contains(...) OR col LIKE '%...%')
   NEVER write: WHERE is_hospital = true AND array_contains(...) OR LIKE   ← WRONG
6. Table columns: use operatorTypeId (camelCase, NOT operatortypeid lowercase)
7. For disease counts: use has_infectious_disease column (NOT specialty name text)
8. LIMIT 50 always
9. Do NOT output python/markdown/explanations — output ONLY the SQL

SPECIALTY EXACT NAMES (camelCase, case-sensitive):
  emergencyMedicine, gynecologyAndObstetrics, generalSurgery, internalMedicine,
  familyMedicine, pediatrics, cardiology, orthopedicSurgery, dentistry,
  ophthalmology, radiology, infectiousDiseases (NOTE: with 's'),
  criticalCareMedicine, psychiatry, anesthesia, nephrology, medicalOncology

FOR Q7.5-STYLE QUERIES (procedures per facility):
  Use procedure_count column for counts — NOT GROUP BY procedure_enriched
  Example: SELECT region_normalised, COUNT(*) as facilities, AVG(procedure_count) as avg_proc
           FROM gold_idp_enriched WHERE procedure_count > 0 GROUP BY region_normalised

FOR GEOGRAPHIC QUERIES: use region_normalised (NOT region) in gold_idp_enriched
FOR DESERT TABLE: use region (NOT region_normalised) and mds_label (NOT desert_label)
"""

# COMMAND ----------
# MAGIC %md ## 6. Router (B7 Fixed: Q7.6 routing, expanded heuristics)

# COMMAND ----------

ROUTER_SYSTEM_PROMPT = """Route a Ghana healthcare query to the correct agent(s).
Return ONLY valid JSON, nothing else.

AGENTS:
  sql       - counts, aggregations, rankings, filtering on structured columns
  rag       - free-text facility lookup, service descriptions, capability search
  geo       - distance calculation, facilities within X km, geographic cold spots
  anomaly   - suspicious claims, data inconsistency, unrealistic ratios
  desert    - medical deserts, underserved regions, specialty gaps, resource allocation
  medical   - clinical reasoning, workforce analysis, service classification
  planning  - NGO action plans, intervention recommendations, resource deployment
  ngo       - NGO coverage, international organization analysis, gap analysis
  general   - definitions, explanations, context

For complex multi-step queries, list sub_agents in execution order.

Return:
{"agent": "primary_agent", "sub_agents": ["agent1","agent2"], "confidence": 0.9,
 "extracted_params": {"region": null, "specialty": null, "facility_type": null, "radius_km": null}}"""


def router_node(state: AgentState) -> AgentState:
    q  = state["query"].lower()
    sc = state.get("step_citations", [])

    HEURISTICS: List[Tuple[str, List[str]]] = [
        ("geo",      [r"within\s*\d+\s*km", r"km\s+of\b", r"\bradius\b", r"\bnearby\b",
                      r"cold\s+spot", r"distance\s+from", r"travel\s+time",
                      r"geographic.{0,20}gap", r"coverage\s+gap\s+map"]),
        ("anomaly",  [r"suspicious", r"flagged", r"anomaly", r"unrealistic",
                      r"false\s+claim", r"inflated", r"inconsistent",
                      r"misrepresent", r"ghost\s+facility",
                      r"procedure.*breadth.*infra", r"claims.*without",
                      r"high.*procedure.*minimal", r"large.*bed.*no.*surg",
                      r"shouldn.t\s+move\s+together", r"things.*move.*together"]),
        ("desert",   [r"medical\s+desert", r"underserved", r"specialty\s+gap",
                      r"region\s+ranking", r"most\s+underserved",
                      r"critical\s+desert", r"no\s+icu\s+region", r"no\s+surgery\s+region",
                      r"missing\s+specialist",
                      # B7 FIX: Q7.6 oversupply/scarcity → desert+medical
                      r"oversupply.*procedure", r"scarcity.*complex",
                      r"concentration.*simple.*procedure", r"vs\s+scarcity"]),
        # B7 FIX: medical heuristics now include oversupply patterns
        ("medical",  [r"workforce.*practi", r"where.*specialist.*practic",
                      r"visiting.*surgeon", r"referral.*language",
                      r"misrepresent\w+", r"subspecialt.*infrastruc",
                      r"oversupply.*vs.*scarcity", r"procedure.*depend.*few",
                      r"service.*classi"]),
        ("planning", [r"action\s+plan", r"intervention", r"deploy.*specialist",
                      r"ngo.*should", r"recommendation", r"resource\s+allocation",
                      r"what\s+should.*do"]),
        ("ngo",      [r"\bngo\b", r"international\s+org", r"ngos\s+in",
                      r"overlap.*ngo", r"coordination.*ngo",
                      r"charity.*gap", r"gaps.*international"]),
        ("sql",      [r"how\s+many", r"count\s+of", r"\btotal\b", r"number\s+of",
                      r"percentage", r"list\s+all", r"rank.*region",
                      r"top.*region", r"which\s+region.*most",
                      r"average", r"correlation\s+between",
                      r"procedure.*depend", r"facilities.*claim"]),
        ("rag",      [r"services.*offer", r"what.*offer", r"korle\s+bu",
                      r"teaching\s+hospital", r"find\s+facilit",
                      r"dialysis", r"what.*provide", r"\bhiv\b", r"malaria.*treat"]),
    ]

    primary_type: Optional[str] = None
    sub_agents: List[str] = []

    # Multi-agent rules for complex queries
    MULTI_AGENT_RULES: List[Tuple[str, str, List[str]]] = [
        # (pattern, primary, sub_agents)
        (r"hospitals.*within\s+\d+\s*km|within\s+\d+\s*km.*hospital", "geo",  ["sql", "medical"]),
        (r"unrealistic.*procedure|procedure.*breadth.*infra|high.*procedure.*minimal", "anomaly", ["medical"]),
        (r"shouldn.t\s+move\s+together|things.*move.*together|large.*bed.*no.*surg", "anomaly", ["medical"]),
        (r"workforce.*practicing|where.*specialist.*practic|surgeon.*practic", "sql", ["medical"]),
        # B7 FIX: Q7.6 "oversupply vs scarcity" → sql + medical (not desert alone)
        (r"oversupply.*vs.*scarcity|scarcity.*complex.*procedure|concentration.*simple", "sql", ["medical"]),
        (r"procedure.*depend.*few|which.*procedure.*few\s+facilit", "sql", ["medical"]),
        (r"ngo.*gap.*need|gaps.*international.*map|no.*ngo.*despite", "ngo", ["desert"]),
    ]

    for pattern, primary, subs in MULTI_AGENT_RULES:
        if re.search(pattern, q, re.IGNORECASE):
            primary_type = primary
            sub_agents   = subs
            break

    if not primary_type:
        for agent, signals in HEURISTICS:
            if any(re.search(sig, q) for sig in signals):
                primary_type = agent
                break

    if not primary_type:
        resp = call_llama(
            [{"role": "user", "content": state["query"]}],
            system_prompt=ROUTER_SYSTEM_PROMPT,
            max_tokens=120, temperature=0.0,
        )
        parsed = parse_json_llm(resp)
        primary_type = parsed.get("agent", "rag")
        if not sub_agents:
            sub_agents = [a for a in parsed.get("sub_agents", []) if a != primary_type]
        if primary_type not in {"sql","rag","geo","anomaly","desert","medical","planning","ngo","map","general"}:
            primary_type = "rag"

    return {
        **state,
        "query_type": primary_type,
        "sub_agents": sub_agents,
        "steps": [f"Router → primary='{primary_type}' chain={sub_agents}"],
        "step_citations": sc + [{
            "step_name":    "router",
            "input_data":   state["query"],
            "output_data":  f"{primary_type} + {sub_agents}",
            "data_sources": ["heuristic_rules"],
            "confidence":   0.9,
        }],
    }

# COMMAND ----------
# MAGIC %md ## 7. SQL Node (B1+B3+B4+B5 Fixed)

# COMMAND ----------

def sql_node(state: AgentState) -> AgentState:
    """B1+B3+B4+B5 Fixed SQL node."""
    with mlflow.start_run(nested=True, run_name="sql_node") as run:
        mlflow.set_tag("step", "sql_node")
        mlflow.set_tag("query", state["query"][:200])

        sql_raw   = call_llama(
            [{"role": "user", "content": f"Question: {state['query']}\n\nSQL:"}],
            system_prompt=SQL_GEN_SYSTEM_PROMPT, max_tokens=600, temperature=0.0,
        )

        # B1 FIX: _clean_sql now properly extracts SELECT block
        sql_clean = _clean_sql(sql_raw)
        # B3 FIX: normalize specialty names and column cases
        sql_clean = _normalize_sql_column_names(sql_clean)
        # B5 FIX: guard against GROUP BY on JSON columns
        json_fix  = _guard_json_group_by(sql_clean)
        if json_fix:
            sql_clean = json_fix
            mlflow.set_tag("json_groupby_fixed", "true")

        mlflow.log_text(sql_clean, "generated_sql.sql")

        try:
            result_pd   = spark.sql(sql_clean).limit(50).toPandas()
            result_json = result_pd.to_json(orient="records", indent=2, default_handler=str)
            step_msg    = f"SQL: {len(result_pd)} rows returned"
            mlflow.log_metric("rows_returned", len(result_pd))
            mlflow.log_text(result_json[:3000], "sql_result.json")
            tables_used = re.findall(r"FROM\s+([\w.]+)", sql_clean, re.IGNORECASE)
        except Exception as e:
            result_json = json.dumps({"error": str(e), "sql": sql_clean})
            step_msg    = f"SQL failed: {str(e)[:120]}"
            tables_used = []
            mlflow.set_tag("sql_error", str(e)[:300])

    # B6 FIX: consume 'sql' from sub_agents so chaining doesn't re-run it
    remaining = [a for a in state.get("sub_agents", []) if a != "sql"]

    return {
        **state,
        "sql_results": result_json,
        "sub_agents":  remaining,
        "steps":       [step_msg],
        "step_citations": state.get("step_citations", []) + [{
            "step_name":    "sql_node",
            "input_data":   state["query"],
            "output_data":  result_json[:300],
            "data_sources": tables_used,
            "confidence":   0.85,
        }],
    }

# COMMAND ----------
# MAGIC %md ## 8. RAG Node (with metadata pre-filtering)

# COMMAND ----------

_rag_index_cache: Optional[Any]  = None
_rag_store_cache: Optional[Dict] = None


def _load_rag(force: bool = False) -> Tuple[Any, Dict]:
    global _rag_index_cache, _rag_store_cache
    if _rag_index_cache is not None and not force:
        return _rag_index_cache, _rag_store_cache
    for idx_path, meta_path in [(cfg.RAG_INDEX_PATH, cfg.RAG_META_PATH),
                                  (cfg.VOLUME_INDEX,  cfg.VOLUME_META)]:
        try:
            idx = faiss.read_index(idx_path)
            with open(meta_path, encoding="utf-8") as f:
                store = json.load(f)
            _rag_index_cache = idx
            _rag_store_cache = store
            print(f"RAG loaded: {idx.ntotal:,} vectors from {idx_path}")
            return idx, store
        except Exception:
            continue
    raise RuntimeError("RAG index not found. Run 05_rag_build_index first.")


def _embed_query(query: str) -> np.ndarray:
    headers = {"Authorization": f"Bearer {cfg.DATABRICKS_TOKEN}", "Content-Type": "application/json"}
    resp = requests.post(cfg.EMBED_ENDPOINT, headers=headers,
                         json={"input": [query[:2000]]}, timeout=30)
    resp.raise_for_status()
    vec = np.array([resp.json()["data"][0]["embedding"]], dtype=np.float32)
    faiss.normalize_L2(vec)
    return vec


def _extract_filters_from_query(query: str) -> Dict:
    q         = query.lower()
    region    = None
    fac_type  = None
    specialty = None
    volunteer = False

    REGIONS = ["greater accra", "ashanti", "western", "eastern", "central", "volta",
               "northern", "upper east", "upper west", "oti", "bono east", "ahafo",
               "savannah", "north east", "western north", "brong-ahafo"]
    for r in REGIONS:
        if r in q:
            region = r.title()
            break

    for t in ["hospital", "clinic", "ngo", "pharmacy"]:
        if t in q:
            fac_type = t
            break

    SPEC_MAP = {
        "dialysis": "nephrology", "cardiology": "cardiology", "surgery": "generalSurgery",
        "emergency": "emergencyMedicine", "obstetric": "gynecologyAndObstetrics",
        "maternity": "gynecologyAndObstetrics", "pediatric": "pediatrics",
        "child": "pediatrics", "icu": "criticalCareMedicine", "radiology": "radiology",
        "hiv": "infectiousDiseases", "malaria": "infectiousDiseases",
        "infectious": "infectiousDiseases", "mental": "psychiatry",
    }
    for kw, spec in SPEC_MAP.items():
        if kw in q:
            specialty = spec
            break

    volunteer = any(w in q for w in ["volunteer", "accepts volunteer"])
    return {"region": region, "facility_type": fac_type, "specialty": specialty, "volunteer": volunteer}


def rag_node(state: AgentState) -> AgentState:
    with mlflow.start_run(nested=True, run_name="rag_node") as run:
        mlflow.set_tag("step", "rag_node")
        try:
            idx, store = _load_rag()
            q_vec      = _embed_query(state["query"])
            filters    = _extract_filters_from_query(state["query"])

            retrieve_k         = min(60, idx.ntotal)
            distances, indices = idx.search(q_vec, retrieve_k)

            results: List[Dict] = []
            for dist, i in zip(distances[0], indices[0]):
                if not (0 <= i < len(store["documents"])):
                    continue
                meta = store["metadata"][i]

                if filters["region"] and meta.get("region", "").lower() != filters["region"].lower():
                    continue
                if filters["facility_type"] and meta.get("facility_type", "").lower() != filters["facility_type"]:
                    continue
                if filters["specialty"]:
                    specs_lower = [s.lower() for s in meta.get("specialties", [])]
                    doc_text    = store["documents"][i].lower()
                    if filters["specialty"].lower() not in specs_lower and \
                       filters["specialty"].lower().replace("and","").replace(" ","") not in doc_text:
                        continue
                if filters["volunteer"] and not meta.get("accepts_volunteers", False):
                    continue

                results.append({
                    "score":    float(dist),
                    "document": store["documents"][i],
                    "metadata": meta,
                    "citations": [c for c in meta.get("idp_citations", []) if len(c) > 15],
                })
                if len(results) >= 8:
                    break

            mlflow.log_metric("docs_retrieved", len(results))
            step_msg = f"RAG: {len(results)} results (filters={filters})"
        except Exception as e:
            results  = []
            step_msg = f"RAG failed: {e}"
            mlflow.set_tag("rag_error", str(e)[:200])

    remaining = [a for a in state.get("sub_agents", []) if a != "rag"]  # B6 FIX

    return {
        **state,
        "rag_results": results,
        "sub_agents":  remaining,
        "steps":       [step_msg],
        "step_citations": state.get("step_citations", []) + [{
            "step_name":    "rag_node",
            "input_data":   state["query"],
            "output_data":  f"{len(results)} docs",
            "data_sources": [r["metadata"].get("unique_id","") for r in results[:4]],
            "confidence":   0.80,
        }],
    }

# COMMAND ----------
# MAGIC %md ## 9. Geo Node (B10 Fixed: coordinates now properly assigned)

# COMMAND ----------

def geo_node(state: AgentState) -> AgentState:
    """B10 Fixed: center coordinates now correctly applied after city lookup."""
    with mlflow.start_run(nested=True, run_name="geo_node") as run:
        mlflow.set_tag("step", "geo_node")
        try:
            q = state["query"].lower()

            # Extract radius
            radius_match = re.search(r"(\d+)\s*km", q)
            radius_km    = float(radius_match.group(1)) if radius_match else 50.0

            # B10 FIX: Sort by descending length to match longest city name first
            center_name  = None
            center_lat, center_lon = 5.5600, -0.1969  # default = Accra
            for city in sorted(cfg.GHANA_CITY_COORDS.keys(), key=len, reverse=True):
                if city in q:
                    center_name = city
                    # B10 FIX: explicitly assign coords after finding city
                    center_lat, center_lon = cfg.GHANA_CITY_COORDS[city]
                    break
            if not center_name:
                center_name = "accra"

            # Load facility coordinates
            src = cfg.IDP_TABLE
            try:
                spark.table(src).limit(1).collect()
            except Exception:
                src = cfg.GOLD_FALLBACK

            sel_cols = ["unique_id","name","facility_type_clean","region_normalised",
                        "city_clean","latitude","longitude",
                        "has_emergency_medicine","has_surgery","has_icu",
                        "has_obstetrics","has_radiology","has_infectious_disease",
                        "has_pediatrics","data_completeness_score",
                        "medical_desert_score","desert_label",
                        "capability_enriched","specialties_enriched","procedure_enriched"]
            avail = set(spark.table(src).columns)
            sel_cols = [c for c in sel_cols if c in avail]

            fac_pd = spark.table(src).select(*sel_cols).toPandas()
            fac_pd["latitude"]  = pd.to_numeric(fac_pd["latitude"],  errors="coerce")
            fac_pd["longitude"] = pd.to_numeric(fac_pd["longitude"], errors="coerce")
            fac_pd = fac_pd.dropna(subset=["latitude","longitude"])
            # Remove centroid-only rows (exact Ghana centroid = no real geocoding)
            fac_pd = fac_pd[~((fac_pd["latitude"] == 7.9465) & (fac_pd["longitude"] == -1.0232))]

            fac_pd["distance_km"] = fac_pd.apply(
                lambda r: haversine_km(center_lat, center_lon, r["latitude"], r["longitude"]),
                axis=1
            )
            within_radius = fac_pd[fac_pd["distance_km"] <= radius_km].copy().sort_values("distance_km")

            # Specialty filter
            SPECIALTY_KEYWORDS = {
                "malaria": "has_infectious_disease", "hiv": "has_infectious_disease",
                "tb": "has_infectious_disease", "infectious": "has_infectious_disease",
                "surgery": "has_surgery", "emergency": "has_emergency_medicine",
                "icu": "has_icu", "obstetric": "has_obstetrics",
                "maternity": "has_obstetrics", "radiology": "has_radiology",
                "pediatric": "has_pediatrics", "x-ray": "has_radiology",
            }
            specialty_filter_col = None
            for kw, col in SPECIALTY_KEYWORDS.items():
                if kw in q:
                    specialty_filter_col = col
                    break

            if specialty_filter_col and specialty_filter_col in within_radius.columns:
                specialty_filtered = within_radius[within_radius[specialty_filter_col] == True]
            else:
                specialty_filtered = within_radius

            # Cold spot detection
            cold_spots: List[Dict] = []
            if any(p in q for p in ["cold spot", "absent", "no surgery", "no emergency"]):
                target = "has_surgery" if "surgery" not in q else "has_emergency_medicine"
                target = "has_emergency_medicine" if "emergency" in q else target
                target = "has_icu" if "icu" in q else target

                if target in fac_pd.columns:
                    region_coverage = fac_pd.groupby("region_normalised")[target].any().reset_index()
                    cold_spot_regions = region_coverage[~region_coverage[target]]["region_normalised"].tolist()
                    for rg in cold_spot_regions:
                        rf = fac_pd[fac_pd["region_normalised"] == rg]
                        if len(rf) > 0:
                            cold_spots.append({
                                "region": rg, "missing_capability": target.replace("has_",""),
                                "facility_count": len(rf),
                                "centroid_lat": round(rf["latitude"].mean(), 4),
                                "centroid_lon": round(rf["longitude"].mean(), 4),
                            })

            results_list = []
            for _, r in specialty_filtered.head(20).iterrows():
                row_d = r.to_dict()
                row_d["distance_km"] = round(float(row_d.get("distance_km", 0)), 2)
                results_list.append(row_d)

            geo_result = {
                "center":           {"name": center_name, "lat": center_lat, "lon": center_lon},
                "radius_km":        radius_km,
                "total_in_radius":  len(within_radius),
                "specialty_matches":len(specialty_filtered),
                "specialty_filter": specialty_filter_col,
                "facilities":       results_list[:15],
                "cold_spots":       cold_spots,
            }
            step_msg = (f"Geo: {len(specialty_filtered)} facilities within {radius_km}km "
                        f"of {center_name.title()}; {len(cold_spots)} cold spots")
            mlflow.log_metric("facilities_in_radius", len(within_radius))
            mlflow.log_metric("specialty_matches",    len(specialty_filtered))
        except Exception as e:
            geo_result = {"error": str(e)}
            step_msg   = f"Geo failed: {e}"
            mlflow.set_tag("geo_error", str(e)[:200])

    remaining = [a for a in state.get("sub_agents", []) if a != "geo"]  # B6 FIX

    return {
        **state,
        "geo_results": geo_result,
        "sub_agents":  remaining,
        "steps":       [step_msg],
        "step_citations": state.get("step_citations", []) + [{
            "step_name":    "geo_node",
            "input_data":   state["query"],
            "output_data":  step_msg,
            "data_sources": [cfg.IDP_TABLE],
            "confidence":   0.90,
        }],
    }

# COMMAND ----------
# MAGIC %md ## 10. Map Node

# COMMAND ----------

def map_node(state: AgentState) -> AgentState:
    with mlflow.start_run(nested=True, run_name="map_node") as run:
        mlflow.set_tag("step", "map_node")
        try:
            src = cfg.IDP_TABLE
            try:
                spark.table(src).limit(1).collect()
            except Exception:
                src = cfg.GOLD_FALLBACK

            exist  = set(spark.table(src).columns)
            wanted = ["unique_id","name","facility_type_clean","organization_type_clean",
                       "city_clean","region_normalised","latitude","longitude",
                       "capacity_int","number_doctors_int","capability_is_valid",
                       "accepts_volunteers_bool","data_completeness_score",
                       "total_stat_anomalies","is_hospital","is_ngo",
                       "has_emergency_medicine","has_surgery","has_icu","has_obstetrics",
                       "medical_desert_score","desert_label"]
            sel_cols = [c for c in wanted if c in exist]
            f_pd    = spark.table(src).select(*sel_cols).toPandas()

            try:
                d_pd      = spark.table(cfg.DESERT_TABLE).select("region","medical_desert_score","mds_label").toPandas()
                desert_map = dict(zip(d_pd["region"], d_pd["medical_desert_score"]))
                label_map  = dict(zip(d_pd["region"], d_pd["mds_label"]))
            except Exception:
                desert_map, label_map = {}, {}

            def _color(mds: float) -> str:
                if mds > 0.75: return "#dc2626"
                if mds > 0.55: return "#ea580c"
                if mds > 0.35: return "#ca8a04"
                if mds > 0.15: return "#16a34a"
                return "#2563eb"

            features = []
            for _, row in f_pd.iterrows():
                lat = safe_float(row.get("latitude"), 0.0)
                lon = safe_float(row.get("longitude"), 0.0)
                if lat == 0.0 and lon == 0.0:
                    continue
                # Skip centroid-fallback rows for cleaner map
                if abs(lat - 7.9465) < 0.001 and abs(lon + 1.0232) < 0.001:
                    continue
                region = safe_str(row.get("region_normalised"), "Unknown")
                mds    = safe_float(desert_map.get(region) or row.get("medical_desert_score"), 0.5)
                label  = label_map.get(region) or safe_str(row.get("desert_label"), "Unknown")

                features.append({
                    "type": "Feature",
                    "geometry": {"type": "Point", "coordinates": [lon, lat]},
                    "properties": {
                        "id": safe_str(row.get("unique_id")), "name": safe_str(row.get("name")),
                        "type": safe_str(row.get("facility_type_clean")).lower(),
                        "city": safe_str(row.get("city_clean")), "region": region,
                        "capacity": safe_int(row.get("capacity_int")),
                        "doctors":  safe_int(row.get("number_doctors_int")),
                        "desert_score": round(mds, 4), "desert_label": label,
                        "has_anomaly":       not bool(row.get("capability_is_valid", True)),
                        "accepts_volunteers": bool(row.get("accepts_volunteers_bool")),
                        "completeness":       round(safe_float(row.get("data_completeness_score")), 3),
                        "has_emergency":      bool(row.get("has_emergency_medicine")),
                        "has_surgery":        bool(row.get("has_surgery")),
                        "has_icu":            bool(row.get("has_icu")),
                        "color": _color(mds),
                        "tooltip": f"{safe_str(row.get('name'))} | {safe_str(row.get('facility_type_clean'))} | {region}",
                    },
                })

            geojson  = {"type": "FeatureCollection", "features": features}
            step_msg = f"Map: {len(features):,} markers"
            mlflow.log_metric("map_features", len(features))
        except Exception as e:
            geojson  = {"type": "FeatureCollection", "features": []}
            step_msg = f"Map failed: {e}"

    remaining = [a for a in state.get("sub_agents", []) if a != "map"]  # B6 FIX

    return {
        **state, "map_data": geojson, "sub_agents": remaining, "steps": [step_msg],
        "step_citations": state.get("step_citations", []) + [{
            "step_name": "map_node", "input_data": state["query"],
            "output_data": step_msg, "data_sources": [src], "confidence": 0.95,
        }],
    }

# COMMAND ----------
# MAGIC %md ## 11. Anomaly Node (B2+B9 Fixed: correct column names, safe fallback)

# COMMAND ----------

def anomaly_node(state: AgentState) -> AgentState:
    """B2 Fixed: uses total_anomaly_flags (not total_stat_anomalies) for anomaly table.
       B9 Fixed: gold_anomaly_report query is fully optional with try/except."""
    with mlflow.start_run(nested=True, run_name="anomaly_node") as run:
        mlflow.set_tag("step", "anomaly_node")
        try:
            anom_df    = spark.table(cfg.ANOMALY_TABLE)
            avail_cols = set(anom_df.columns)

            # B2 FIX: gold_anomaly_flags uses total_anomaly_flags (not total_stat_anomalies)
            want_cols = [c for c in [
                "unique_id", "name", "city_clean", "region_normalised", "facility_type_clean",
                "total_anomaly_flags",          # B2 FIX: correct col for anomaly table
                "anomaly_risk_level",
                "llm_anomaly_severity",
                "llm_priority_action",
                "llm_data_quality_score",
                "llm_confirmed_anomaly_count",
                "llm_clinical_assessment",
                "capability_anomalies",
                "procedure_count", "equipment_count", "capability_count",
                "data_completeness_score",
                "medical_desert_score", "desert_label",
                "stat_anomaly_capability_inflation",
                "stat_anomaly_hospital_no_doctors",
                "stat_anomaly_ghost_facility",
                "stat_anomaly_procedure_breadth",
                "enhanced_procedures_no_equipment",
                "enhanced_type_capability_mismatch",
                "enhanced_icu_no_infrastructure",
            ] if c in avail_cols]

            anomalies = (
                anom_df.select(*want_cols)
                .filter(F.col("anomaly_risk_level").isin("CRITICAL", "HIGH", "MEDIUM"))
                .orderBy(F.desc("total_anomaly_flags"))
                .limit(30)
                .toPandas()
                .infer_objects(copy=False)
                .fillna("")
                .to_dict(orient="records")
            )

            # B9 FIX: gold_anomaly_report is optional — don't crash if absent
            report: List[Dict] = []
            try:
                report = (spark.table(cfg.ANOMALY_REPORT)
                    .orderBy(F.desc("flag_rate")).limit(10)
                    .toPandas().infer_objects(copy=False).fillna("").to_dict(orient="records"))
            except Exception:
                pass   # table may not exist — that's OK

            step_msg = f"Anomaly: {len(anomalies)} flagged facilities"
            mlflow.log_metric("anomalies_returned", len(anomalies))

        except Exception as e:
            # Fallback: query IDP table with its own anomaly column name
            try:
                # B2 FIX: IDP table uses total_stat_anomalies (NOT total_anomaly_flags)
                anom_df2 = spark.table(cfg.IDP_TABLE).filter(
                    (F.col("total_stat_anomalies") > 0) |
                    (F.col("capability_is_valid") == False)
                )
                anomalies = (anom_df2.select(
                    "name","city_clean","region_normalised","facility_type_clean",
                    "total_stat_anomalies","capability_anomalies",
                    "stat_anomaly_capability_inflation","stat_anomaly_hospital_no_doctors",
                    "stat_anomaly_procedure_breadth","capability_confidence",
                    "procedure_count","equipment_count"
                ).orderBy(F.desc("total_stat_anomalies")).limit(30)
                .toPandas().infer_objects(copy=False).fillna("").to_dict(orient="records"))
                report    = []
                step_msg  = f"Anomaly (IDP fallback): {len(anomalies)} rows"
            except Exception as e2:
                anomalies = []
                report    = []
                step_msg  = f"Anomaly failed: {e2}"

    remaining = [a for a in state.get("sub_agents", []) if a != "anomaly"]  # B6 FIX

    return {
        **state,
        "anomalies": anomalies,
        "sub_agents": remaining,
        "steps":     [step_msg],
        "step_citations": state.get("step_citations", []) + [{
            "step_name":    "anomaly_node",
            "input_data":   state["query"],
            "output_data":  f"{len(anomalies)} anomalies",
            "data_sources": [cfg.ANOMALY_TABLE],
            "confidence":   0.85,
        }],
    }

# COMMAND ----------
# MAGIC %md ## 12. Desert Node

# COMMAND ----------

def desert_node(state: AgentState) -> AgentState:
    with mlflow.start_run(nested=True, run_name="desert_node") as run:
        mlflow.set_tag("step", "desert_node")
        try:
            desert_pd = spark.table(cfg.DESERT_TABLE).toPandas()
            desert_pd = desert_pd.sort_values("medical_desert_score", ascending=False)
            desert_pd = desert_pd.infer_objects(copy=False)

            try:
                reg_pd = spark.table(cfg.REGIONAL_TABLE).toPandas().infer_objects(copy=False)
                for col in ["missing_critical_specialties","recommended_actions","all_specialties"]:
                    if col in reg_pd.columns:
                        reg_pd[col] = reg_pd[col].apply(ensure_list)
                reg_dict = reg_pd.to_dict(orient="records")
            except Exception:
                reg_dict = []

            cold_spots = []
            for _, r in desert_pd.iterrows():
                missing = ensure_list(r.get("critical_specialties_missing","[]"))
                if len(missing) >= 4:
                    cold_spots.append({
                        "region":            r["region"],
                        "mds_label":         r["mds_label"],    # correct col name
                        "mds_score":         round(float(r["medical_desert_score"]), 4),
                        "missing_specialties": missing,
                        "centroid_lat":      r.get("centroid_lat"),
                        "centroid_lon":      r.get("centroid_lon"),
                        "recommended_actions": ensure_list(r.get("recommended_actions","[]"))[:3],
                    })

            desert_data = {
                "all_regions":       desert_pd.to_dict(orient="records"),
                "most_underserved":  desert_pd.head(5).to_dict(orient="records"),
                "least_underserved": desert_pd.tail(3).to_dict(orient="records"),
                "cold_spots":        cold_spots,
                "regional_summary":  reg_dict,
                "critical_count":    int((desert_pd["mds_label"] == "Critical Desert").sum())
                                     if "mds_label" in desert_pd.columns else 0,
                "severe_count":      int((desert_pd["mds_label"] == "Severe Desert").sum())
                                     if "mds_label" in desert_pd.columns else 0,
            }
            step_msg = (f"Desert: {len(desert_pd)} regions; "
                        f"{desert_data['critical_count']} Critical, "
                        f"{desert_data['severe_count']} Severe; "
                        f"{len(cold_spots)} cold spots")
            mlflow.log_metric("regions_loaded", len(desert_pd))

        except Exception as e:
            desert_data = {"error": str(e), "scores": []}
            step_msg    = f"Desert failed: {e}"

    remaining = [a for a in state.get("sub_agents", []) if a != "desert"]  # B6 FIX

    return {
        **state, "desert_data": desert_data, "sub_agents": remaining, "steps": [step_msg],
        "step_citations": state.get("step_citations", []) + [{
            "step_name": "desert_node", "input_data": state["query"],
            "output_data": step_msg,
            "data_sources": [cfg.DESERT_TABLE, cfg.REGIONAL_TABLE], "confidence": 0.90,
        }],
    }

# COMMAND ----------
# MAGIC %md ## 13. Medical Reasoning Node

# COMMAND ----------

MEDICAL_REASONING_SYSTEM_PROMPT = """You are a senior medical advisor specializing in
sub-Saharan African healthcare systems, specifically Ghana.

You receive structured evidence from facility databases and must provide:
1. Clinical interpretation of patterns and anomalies
2. Infrastructure assessment: do stated capabilities match real operational capacity?
3. Workforce distribution analysis: where specialists are actually practicing
4. Service classification: permanent vs itinerant/outreach vs referral
5. Misrepresentation risk assessment: what claims are clinically implausible?
6. Actionable recommendations for the Virtue Foundation field team

Clinical reasoning rules:
- "Things that shouldn't move together": large bed count but zero surgical equipment = high suspicion
- Clinic claiming ICU with no equipment is implausible in Ghana context
- Hospital with procedure_count>8 but equipment_count=0 indicates data over-reporting
- In Ghana: regional hospitals serve 100,000–500,000 population; district hospitals 50,000–100,000
- A genuine surgical center needs: operating theatre, oxygen supply, anaesthesia, ≥2 doctors minimum
- For oversupply/scarcity: simple procedures (antenatal, immunization) are widespread;
  complex ones (cardiac surgery, neurosurgery) concentrate in 2-3 tertiary centers

Ground all statements in the provided data. State uncertainty explicitly.
Write in plain English for NGO programme officers. Be specific — name facilities and regions."""


def medical_node(state: AgentState) -> AgentState:
    with mlflow.start_run(nested=True, run_name="medical_node") as run:
        mlflow.set_tag("step", "medical_node")
        evidence: List[str] = []

        if state.get("sql_results"):
            try:
                rows = json.loads(state["sql_results"])
                if isinstance(rows, list):
                    evidence.append(f"DATABASE QUERY ({len(rows)} rows):\n"
                                     + json.dumps(rows[:20], indent=2, default=str)[:3000])
                else:
                    evidence.append(f"DATA: {state['sql_results'][:2000]}")
            except Exception:
                evidence.append(f"DATA: {state['sql_results'][:2000]}")

        if state.get("anomalies"):
            evidence.append("ANOMALY FLAGS:\n"
                             + json.dumps(state["anomalies"][:8], indent=2, default=str)[:2500])

        if state.get("geo_results") and not state["geo_results"].get("error"):
            gr = state["geo_results"]
            evidence.append(
                f"GEOSPATIAL: {gr.get('specialty_matches',0)} facilities within "
                f"{gr.get('radius_km',50)}km of {gr.get('center',{}).get('name','?')}\n"
                + json.dumps(gr.get("facilities",gr.get("cold_spots",[]))[:8], indent=2, default=str)[:2000]
            )

        if state.get("desert_data"):
            dd = state["desert_data"]
            if dd.get("most_underserved"):
                evidence.append("DESERT SCORES:\n"
                                 + json.dumps(dd["most_underserved"][:5], indent=2, default=str)[:1500])

        for r in (state.get("rag_results") or [])[:4]:
            m = r["metadata"]
            evidence.append(
                f"FACILITY: {m.get('name','?')} ({m.get('facility_type','?')}, {m.get('region','?')})\n"
                f"  Procs={len(m.get('procedures',[]))} Equip={len(m.get('equipment',[]))} "
                f"Caps={len(m.get('capabilities',[]))}\n"
                f"  {r['document'][:300]}"
            )

        if not evidence:
            evidence.append("No upstream data. Reasoning from general Ghana healthcare knowledge.")

        prompt = (
            f"Question: {state['query']}\n\n"
            f"Evidence:\n{'---'.join(evidence)[:5000]}\n\n"
            "Provide: (1) Clinical interpretation, (2) Misrepresentation/anomaly assessment, "
            "(3) Workforce/infrastructure gap analysis, (4) 2-3 specific NGO recommendations."
        )

        analysis_text = call_llama(
            [{"role": "user", "content": prompt}],
            system_prompt=MEDICAL_REASONING_SYSTEM_PROMPT,
            max_tokens=1200, temperature=0.2,
        )

        medical_analysis = {"reasoning": analysis_text, "evidence_sources": len(evidence)}
        mlflow.log_text(analysis_text, "medical_reasoning.txt")
        step_msg = f"Medical reasoning: {len(analysis_text)} chars"

    remaining = [a for a in state.get("sub_agents", []) if a != "medical"]  # B6 FIX

    return {
        **state,
        "medical_analysis": medical_analysis,
        "sub_agents":       remaining,
        "steps":            [step_msg],
        "step_citations": state.get("step_citations", []) + [{
            "step_name":    "medical_node",
            "input_data":   f"Evidence from {len(evidence)} sources",
            "output_data":  analysis_text[:200],
            "data_sources": ["llama_medical_reasoning"],
            "confidence":   0.75,
        }],
    }

# COMMAND ----------
# MAGIC %md ## 14. Planning Node (PDF Core Feature 3)

# COMMAND ----------

PLANNING_SYSTEM_PROMPT = """You are a healthcare planning advisor for the Virtue Foundation.
Generate clear, actionable intervention plans for non-technical NGO planners.

Rules:
- Write in plain English — no SQL, no column names, no technical jargon
- Be specific: name regions, name facilities, cite numbers
- Prioritise by urgency: immediate (0–30 days) / medium (3 months) / long-term (12 months)
- Format as a numbered action plan
- Include: (a) what the gap is, (b) who is affected, (c) what action to take, (d) expected impact"""


def planning_node(state: AgentState) -> AgentState:
    with mlflow.start_run(nested=True, run_name="planning_node") as run:
        mlflow.set_tag("step", "planning_node")
        try:
            reg_pd = (spark.table(cfg.REGIONAL_TABLE)
                       .orderBy(F.desc("medical_desert_score")).limit(17)
                       .toPandas().infer_objects(copy=False))
            for col in ["missing_critical_specialties", "recommended_actions"]:
                if col in reg_pd.columns:
                    reg_pd[col] = reg_pd[col].apply(ensure_list)

            try:
                desert_pd     = spark.table(cfg.DESERT_TABLE).toPandas().infer_objects(copy=False)
                desert_lookup = dict(zip(desert_pd["region"], desert_pd["mds_label"]))
            except Exception:
                desert_lookup = {}

            plan_context: List[Dict] = []
            for _, r in reg_pd.iterrows():
                region  = safe_str(r.get("region_normalised","?"))
                missing = ensure_list(r.get("missing_critical_specialties",[]))
                plan_context.append({
                    "region":               region,
                    "mds_label":            desert_lookup.get(region, safe_str(r.get("desert_label","?"))),
                    "medical_desert_score": round(safe_float(r.get("medical_desert_score"),0.5), 3),
                    "total_facilities":     safe_int(r.get("total_facilities")),
                    "hospitals":            safe_int(r.get("hospital_count")),
                    "ngos":                 safe_int(r.get("ngo_count")),
                    "total_doctors":        safe_int(r.get("total_doctors")),
                    "total_beds":           safe_int(r.get("total_beds")),
                    "icu_facilities":       safe_int(r.get("icu_facilities")),
                    "emergency_facilities": safe_int(r.get("emergency_medicine_facilities")),
                    "surgery_facilities":   safe_int(r.get("surgery_facilities")),
                    "missing_specialties":  missing,
                    "volunteer_facilities": safe_int(r.get("volunteer_facilities")),
                })

            prompt = (
                f"User question: {state['query']}\n\n"
                f"Regional health data (sorted by urgency):\n"
                + json.dumps(plan_context, indent=2)[:5000]
                + "\n\nGenerate a specific, numbered action plan."
            )

            plan_text = call_llama(
                [{"role": "user", "content": prompt}],
                system_prompt=PLANNING_SYSTEM_PROMPT,
                max_tokens=1400, temperature=0.2,
            )

            planning_data = {"plan": plan_text, "region_count": len(plan_context)}
            mlflow.log_text(plan_text, "action_plan.txt")
            step_msg = f"Planning: {len(plan_context)} regions assessed"
        except Exception as e:
            planning_data = {"error": str(e), "plan": ""}
            step_msg      = f"Planning failed: {e}"

    remaining = [a for a in state.get("sub_agents", []) if a != "planning"]  # B6 FIX

    return {
        **state,
        "planning_data": planning_data,
        "sub_agents":    remaining,
        "steps":         [step_msg],
        "step_citations": state.get("step_citations", []) + [{
            "step_name":    "planning_node",
            "input_data":   state["query"],
            "output_data":  plan_text[:200] if "plan_text" in dir() else "error",
            "data_sources": [cfg.REGIONAL_TABLE, cfg.DESERT_TABLE],
            "confidence":   0.80,
        }],
    }

# COMMAND ----------
# MAGIC %md ## 15. NGO Node

# COMMAND ----------

def ngo_node(state: AgentState) -> AgentState:
    with mlflow.start_run(nested=True, run_name="ngo_node") as run:
        mlflow.set_tag("step", "ngo_node")
        try:
            avail = set(spark.table(cfg.IDP_TABLE).columns)
            ngo_sel = [c for c in ["name","region_normalised","city_clean",
                                    "ngo_serves_ghana","accepts_volunteers_bool",
                                    "specialties_enriched","capability_enriched",
                                    "data_completeness_score","medical_desert_score","desert_label",
                                    "has_emergency_medicine","has_surgery"] if c in avail]

            ngo_pd = (spark.table(cfg.IDP_TABLE)
                .filter(F.col("is_ngo") == True)
                .select(*ngo_sel)
                .toPandas().infer_objects(copy=False))

            regional_ngo = (spark.table(cfg.REGIONAL_TABLE)
                .select("region_normalised","ngo_count","medical_desert_score",
                         "desert_label","missing_critical_specialties")
                .orderBy(F.desc("medical_desert_score"))
                .toPandas().infer_objects(copy=False))

            for col in ["missing_critical_specialties"]:
                if col in regional_ngo.columns:
                    regional_ngo[col] = regional_ngo[col].apply(ensure_list)

            ngo_gaps = regional_ngo[
                (pd.to_numeric(regional_ngo["ngo_count"], errors="coerce").fillna(0) == 0) &
                (pd.to_numeric(regional_ngo["medical_desert_score"], errors="coerce") > 0.4)
            ].to_dict(orient="records")

            ngo_data = {
                "total_ngos":        len(ngo_pd),
                "ngos":              ngo_pd.head(20).to_dict(orient="records"),
                "regional_coverage": regional_ngo.to_dict(orient="records"),
                "coverage_gaps":     ngo_gaps,
            }
            step_msg = f"NGO: {len(ngo_pd)} NGOs; {len(ngo_gaps)} high-need regions with no NGO coverage"
        except Exception as e:
            ngo_data = {"error": str(e)}
            step_msg = f"NGO failed: {e}"

    remaining = [a for a in state.get("sub_agents", []) if a != "ngo"]  # B6 FIX

    return {
        **state, "ngo_data": ngo_data, "sub_agents": remaining, "steps": [step_msg],
        "step_citations": state.get("step_citations", []) + [{
            "step_name": "ngo_node", "input_data": state["query"],
            "output_data": step_msg,
            "data_sources": [cfg.IDP_TABLE, cfg.REGIONAL_TABLE], "confidence": 0.85,
        }],
    }

# COMMAND ----------
# MAGIC %md ## 16. Synthesiser (B12 Fixed: safe .get() for planning_data)

# COMMAND ----------

SYNTHESISER_SYSTEM_PROMPT = """You are a healthcare intelligence advisor for the Virtue Foundation.
Audience: NGO programme officers — non-technical, action-oriented.

ANSWER FORMAT:
1. Direct answer (numbers, names, regions) — first paragraph
2. Evidence summary (cite specific facilities / regions by name)
3. Clinical context (why this matters for patients — 1-2 sentences)
4. Recommended NGO actions (concrete, specific: who deploys what, where, when)

RULES:
- Never say "some facilities" — always name specific ones from the data
- Use numbers: "7 of 17 regions lack emergency medicine"
- If map data present: "An interactive map with {N} markers has been prepared"
- Integrate medical reasoning conclusions when present — don't repeat raw data
- Include planning action items when present
- Write "According to facility records [facility name]..." for citations
- Do NOT mention SQL, JSON, column names, or internal system details"""


def synthesiser_node(state: AgentState) -> AgentState:
    """B12 Fixed: all dict accesses use safe .get() to avoid KeyError."""
    with mlflow.start_run(nested=True, run_name="synthesiser_node") as run:
        mlflow.set_tag("step", "synthesiser_node")

        evidence: List[str] = []
        facility_citations: List[Dict] = []

        if state.get("sql_results"):
            try:
                rows = json.loads(state["sql_results"])
                if isinstance(rows, list):
                    evidence.append(f"DATA ({len(rows)} records):\n"
                                     + json.dumps(rows[:20], indent=2, default=str)[:3000])
                else:
                    evidence.append(f"DATA: {state['sql_results'][:2000]}")
            except Exception:
                evidence.append(f"DATA: {state['sql_results'][:2000]}")

        for r in (state.get("rag_results") or [])[:6]:
            m = r["metadata"]
            cits = r.get("citations", m.get("idp_citations",[]))[:3]
            cit_str = ("\n  SOURCE: " + " | ".join(c[:100] for c in cits if c)) if cits else ""
            evidence.append(
                f"FACILITY [{m.get('name','?')}] ({m.get('facility_type','?')}, "
                f"{m.get('city','?')}, {m.get('region','?')}) score={r['score']:.3f}:\n"
                f"{r['document'][:350]}{cit_str}"
            )
            facility_citations.append({
                "facility_id":    m.get("unique_id",""),
                "facility_name":  m.get("name",""),
                "facility_type":  m.get("facility_type",""),
                "region":         m.get("region",""),
                "city":           m.get("city",""),
                "relevance_score": r["score"],
                "source_document": r["document"][:200],
                "idp_citations":  cits,
                "idp_run_id":     m.get("idp_run_id",""),
                "desert_label":   m.get("desert_label",""),
                "step_name":      "rag_node",
            })

        if state.get("geo_results") and not (state["geo_results"] or {}).get("error"):
            gr = state["geo_results"]
            evidence.append(
                f"GEOSPATIAL: {gr.get('specialty_matches',0)} facilities within "
                f"{gr.get('radius_km',50)}km of {gr.get('center',{}).get('name','?').title()}\n"
                + json.dumps(gr.get("facilities",[])[:6], indent=2, default=str)[:2500]
            )
            if gr.get("cold_spots"):
                evidence.append("COLD SPOTS:\n"
                                 + json.dumps(gr["cold_spots"], indent=2, default=str)[:1500])

        if state.get("anomalies"):
            evidence.append(f"ANOMALIES (top {min(len(state['anomalies']),5)}):\n"
                             + json.dumps(state["anomalies"][:5], indent=2, default=str)[:2000])

        if state.get("desert_data"):
            dd = state["desert_data"]
            if dd.get("most_underserved"):
                evidence.append("MOST UNDERSERVED:\n"
                                 + json.dumps(dd["most_underserved"][:5], indent=2, default=str)[:1500])
            if dd.get("cold_spots"):
                evidence.append("COLD SPOTS:\n"
                                 + json.dumps(dd["cold_spots"][:5], indent=2, default=str)[:1000])

        if state.get("medical_analysis"):
            ma = state["medical_analysis"]
            # B12 FIX: use .get() not direct key access
            if ma.get("reasoning"):
                evidence.append(f"CLINICAL ANALYSIS:\n{ma['reasoning'][:2000]}")

        if state.get("planning_data"):
            pd_data = state["planning_data"]
            # B12 FIX: use .get() not direct key access
            plan = pd_data.get("plan","")
            if plan:
                evidence.append(f"NGO ACTION PLAN:\n{plan[:2000]}")

        if state.get("ngo_data"):
            nd = state["ngo_data"]
            if nd.get("coverage_gaps"):
                evidence.append("NGO COVERAGE GAPS:\n"
                                 + json.dumps(nd["coverage_gaps"][:5], indent=2, default=str)[:1000])

        if state.get("map_data"):
            n = len((state["map_data"] or {}).get("features", []))
            evidence.append(f"MAP: Interactive GeoJSON map prepared with {n:,} facility markers.")

        if not evidence:
            evidence.append("No data retrieved. Answering from Ghana healthcare domain knowledge.")

        context       = "\n\n---\n\n".join(evidence)
        synth_prompt  = f'User question: "{state["query"]}"\n\nEvidence:\n{context[:6500]}'

        answer = call_llama(
            [{"role": "user", "content": synth_prompt}],
            system_prompt=SYNTHESISER_SYSTEM_PROMPT,
            max_tokens=1400, temperature=0.25,
        )

        mlflow.log_text(answer, "final_answer.txt")
        mlflow.log_dict({"citations": facility_citations}, "facility_citations.json")
        mlflow.log_dict({"step_citations": state.get("step_citations",[])}, "step_citations.json")
        mlflow.log_metric("evidence_sources",   len(evidence))
        mlflow.log_metric("facility_citations", len(facility_citations))

    return {
        **state,
        "answer":     answer,
        "citations":  facility_citations,
        "steps":      ["Synthesiser: answer assembled"],
        "step_citations": state.get("step_citations",[]) + [{
            "step_name":    "synthesiser",
            "input_data":   f"{len(evidence)} evidence sources",
            "output_data":  answer[:200],
            "data_sources": ["llama_3.3_70b"] + [c.get("facility_id","") for c in facility_citations[:3]],
            "confidence":   0.80,
        }],
    }

# COMMAND ----------
# MAGIC %md ## 17. Build LangGraph (B6 Fixed: proper chaining via sub_agents consumed in each node)

# COMMAND ----------

def _route_after_node(state: AgentState) -> str:
    """
    B6 FIX: Routing logic after each node.
    Each node already popped its key from sub_agents before returning.
    So we just check what remains and route accordingly.
    """
    remaining = state.get("sub_agents", [])
    if not remaining:
        return "synthesiser"

    next_agent = remaining[0]
    NODE_MAP = {
        "sql":     "sql_query",
        "rag":     "rag_search",
        "geo":     "geo_calc",
        "anomaly": "anomaly_check",
        "desert":  "desert_check",
        "medical": "medical_reason",
        "planning":"planning_sys",
        "ngo":     "ngo_analysis",
        "map":     "map_gen",
    }
    return NODE_MAP.get(next_agent, "synthesiser")


ALL_WORK_NODES = ["sql_query","rag_search","geo_calc","map_gen","anomaly_check",
                   "desert_check","medical_reason","planning_sys","ngo_analysis","synthesiser"]


def build_virtue_agent():
    wf = StateGraph(AgentState)

    wf.add_node("router",         router_node)
    wf.add_node("sql_query",      sql_node)
    wf.add_node("rag_search",     rag_node)
    wf.add_node("geo_calc",       geo_node)
    wf.add_node("map_gen",        map_node)
    wf.add_node("anomaly_check",  anomaly_node)
    wf.add_node("desert_check",   desert_node)
    wf.add_node("medical_reason", medical_node)
    wf.add_node("planning_sys",   planning_node)
    wf.add_node("ngo_analysis",   ngo_node)
    wf.add_node("synthesiser",    synthesiser_node)

    wf.set_entry_point("router")

    PRIMARY_MAP = {
        "sql":     "sql_query",   "rag":     "rag_search",
        "geo":     "geo_calc",    "map":     "map_gen",
        "anomaly": "anomaly_check","desert":  "desert_check",
        "medical": "medical_reason","planning":"planning_sys",
        "ngo":     "ngo_analysis", "general": "rag_search",
    }

    wf.add_conditional_edges(
        "router",
        lambda s: PRIMARY_MAP.get(s["query_type"], "rag_search"),
        {v: v for v in set(PRIMARY_MAP.values())},
    )

    # B6 FIX: All work nodes route through _route_after_node which reads remaining sub_agents
    # Each node already consumed its own key from sub_agents before returning.
    # Possible targets from any node: any other work node + synthesiser
    for node_name in ["sql_query","rag_search","geo_calc","map_gen","anomaly_check",
                       "desert_check","medical_reason","planning_sys","ngo_analysis"]:
        possible_next = [n for n in ALL_WORK_NODES]   # all nodes + synthesiser
        wf.add_conditional_edges(node_name, _route_after_node, {n: n for n in possible_next})

    wf.add_edge("synthesiser", END)
    return wf.compile()


VIRTUE_AGENT = build_virtue_agent()
print("✅ LangGraph v4 agent compiled — 10 nodes + fixed chaining")

# COMMAND ----------
# MAGIC %md ## 18. Agent Runner

# COMMAND ----------

def run_agent(query: str, session_id: str = "demo") -> Dict[str, Any]:
    mlflow.set_experiment(cfg.MLFLOW_EXP)

    with mlflow.start_run(run_name=f"agent_{query[:40].replace(' ','_')}") as run:
        mlflow.set_tag("query",      query[:250])
        mlflow.set_tag("session_id", session_id)
        mlflow.set_tag("pipeline",   "virtue_foundation_langgraph_v4")

        initial = AgentState(
            query=query, session_id=session_id, query_type="", sub_agents=[],
            sql_results=None, rag_results=[], geo_results=None, map_data=None,
            anomalies=[], desert_data=None, medical_analysis=None,
            planning_data=None, ngo_data=None,
            answer="", citations=[], step_citations=[],
            steps=[], run_id=run.info.run_id, error=None,
        )

        result = VIRTUE_AGENT.invoke(initial)

        mlflow.log_param("query_type",           result.get("query_type",""))
        mlflow.log_param("sub_agents_executed",  str(result.get("sub_agents",[])))
        mlflow.log_metric("steps_count",         len(result.get("steps",[])))
        mlflow.log_metric("facility_citations",  len(result.get("citations",[])))
        mlflow.log_metric("step_citations_count",len(result.get("step_citations",[])))
        mlflow.log_text(result.get("answer",""), "final_answer.txt")

        output = {
            "answer":           result.get("answer",""),
            "citations":        result.get("citations",[]),
            "step_citations":   result.get("step_citations",[]),
            "steps":            result.get("steps",[]),
            "query_type":       result.get("query_type",""),
            "sub_agents":       result.get("sub_agents",[]),
            "map_data":         result.get("map_data"),
            "geo_results":      result.get("geo_results"),
            "anomalies":        result.get("anomalies",[]),
            "desert_data":      result.get("desert_data"),
            "medical_analysis": result.get("medical_analysis"),
            "planning_data":    result.get("planning_data"),
            "ngo_data":         result.get("ngo_data"),
            "sql_results":      result.get("sql_results"),
            "mlflow_run_id":    run.info.run_id,
        }

        print(f"\n{'='*70}")
        print(f"Query      : {query}")
        print(f"Type       : {output['query_type']} + remaining_chain={output['sub_agents']}")
        print(f"Steps      : {output['steps']}")
        print(f"Citations  : {len(output['citations'])} facilities / {len(output['step_citations'])} nodes")
        print(f"MLflow     : {run.info.run_id}")
        print(f"{'='*70}")
        print(f"\nAnswer:\n{output['answer']}\n")
        return output

# COMMAND ----------
# MAGIC %md ## 19. Must-Have Query Evaluation Suite

# COMMAND ----------

MUST_HAVE_QUERIES = [
    {"id":"1.1", "q":"How many hospitals have cardiology in Ghana?",               "expected":"sql"},
    {"id":"1.2", "q":"How many hospitals in Ashanti region have surgery?",         "expected":"sql"},
    {"id":"1.3", "q":"What services does Korle Bu Teaching Hospital offer?",       "expected":"rag"},
    {"id":"1.4", "q":"Are there any clinics in Kumasi that do dialysis?",          "expected":"rag"},
    {"id":"1.5", "q":"Which region in Ghana has the most hospitals?",              "expected":"sql"},
    {"id":"2.1", "q":"How many hospitals treating malaria are within 50km of Accra?","expected":"geo"},
    {"id":"2.3", "q":"Where are the largest geographic cold spots where surgery is absent?","expected":"geo"},
    {"id":"4.4", "q":"Which facilities claim an unrealistic number of procedures?",   "expected":"anomaly"},
    {"id":"4.7", "q":"What correlations exist between facility characteristics in Ghana?","expected":"sql"},
    {"id":"4.8", "q":"Which facilities have unusually high procedure breadth vs minimal infrastructure?","expected":"anomaly"},
    {"id":"4.9", "q":"Where do we see things that should not move together such as large bed counts with no surgical equipment?","expected":"anomaly"},
    {"id":"6.1", "q":"Where is the surgical workforce actually practicing in Ghana?",  "expected":"sql"},
    {"id":"7.5", "q":"Which procedures depend on very few facilities in Ghana?",       "expected":"sql"},
    {"id":"7.6", "q":"Where is there oversupply of simple procedures vs scarcity of complex procedures?","expected":"sql"},
]

print("=" * 70)
print("MUST-HAVE QUERY EVALUATION (14 queries)")
print("=" * 70)

eval_results = []
passed = 0

for item in MUST_HAVE_QUERIES:
    qid, query, expected = item["id"], item["q"], item["expected"]
    print(f"\n[{qid}] {query[:60]!r}")
    try:
        result   = run_agent(query, session_id=f"eval_{qid}")
        got_type = result["query_type"]
        match    = got_type == expected
        has_ans  = bool(result["answer"] and len(result["answer"]) > 80)
        has_cits = len(result["citations"]) > 0 or len(result["step_citations"]) > 0

        if has_ans:
            passed += 1

        print(f"  Type    : {'✅' if match else '⚠️'} got='{got_type}' expected='{expected}'")
        print(f"  Answer  : {'✅' if has_ans else '❌'} ({len(result['answer'])} chars)")
        print(f"  Cits    : {len(result['citations'])} facilities, {len(result['step_citations'])} nodes")
        print(f"  MLflow  : {result['mlflow_run_id']}")

        eval_results.append({
            "id": qid, "query": query, "expected": expected, "actual": got_type,
            "type_match": match, "has_answer": has_ans, "has_citations": has_cits,
            "citation_count": len(result["citations"]),
            "step_citation_count": len(result["step_citations"]),
            "run_id": result["mlflow_run_id"],
        })
    except Exception as e:
        print(f"  ❌ FAILED: {e}")
        eval_results.append({"id": qid, "query": query, "error": str(e), "has_answer": False})

print(f"\n{'='*70}")
print(f"EVALUATION: {passed}/{len(MUST_HAVE_QUERIES)} answered ({passed/len(MUST_HAVE_QUERIES)*100:.0f}%)")
print(f"{'='*70}")

try:
    mlflow.set_experiment(cfg.MLFLOW_EXP)
    with mlflow.start_run(run_name="06_evaluation_v4"):
        mlflow.log_metric("queries_answered", passed)
        mlflow.log_metric("pass_rate",        passed / len(MUST_HAVE_QUERIES))
        mlflow.log_dict({"eval_results": eval_results}, "evaluation_v4.json")
    print("Evaluation logged ✅")
except Exception as e:
    print(f"MLflow log skipped: {e}")

print()
print("✅ Notebook 06 v4 complete")
print(f"MLflow experiment: {cfg.MLFLOW_EXP}")
print("Usage: result = run_agent('your question')")